# WeakTr, step by step: a hands-on walkthrough

This notebook walks through **WeakTr: Exploring Plain Vision Transformer for Weakly-supervised Semantic Segmentation** (Zhu et al., IEEE TIP), using the official code in `WeakTr/`.

It assumes **no background**. Every idea gets a short explanation first, then code you can run, then a picture or a printout that shows the idea at work.

**What runs where**

| Part | What you do | Runs on a laptop CPU? |
|---|---|---|
| 0 | Setup, download small files (~180 MB total) | ✅ |
| 1 | The problem: segmentation with only image-level labels | ✅ |
| 2 | Vision Transformer crash course | ✅ |
| 3 | **Phase 1**: end-to-end CAM generation with Adaptive Attention Fusion (AAF), using the authors' trained checkpoint | ✅ (a few seconds per image) |
| 4 | From CAM to pseudo mask, and how mIoU is measured | ✅ (the optional PASCAL VOC evaluation needs the dataset) |
| 5 | **Phase 2**: online retraining with the gradient-clipping decoder, plus a small experiment you can train yourself | ✅ (~5–10 min) |
| 6 | Reproducing the paper's numbers at full scale | ❌ needs a CUDA GPU and Linux; exact commands are given |

**How the code relates to the repo.** The repo depends on old versions of `timm`, `mmcv`, `apex` and `xformers`, plus CUDA, so it won't run on a Windows CPU machine. This notebook therefore **re-implements the important pieces in plain PyTorch, line for line**. Each block says which repo file and lines it copies. The re-implementation loads the authors' official checkpoint with no missing weights, so the CAMs you see are WeakTr's real outputs.

Run the cells from top to bottom (**Run All** works).

Repository copy: use `WEAKTR_CACHE` to override the external cache (default: sibling `../weaktr_cache`). Outputs were cleared on import. Historical narrative and demonstrations are educational, not current reproduction acceptance evidence; use `WeakTr_Kaggle.ipynb` and `docs/KAGGLE.md` for the maintained workflow.


---
# Part 0: Setup

In [ ]:
import math, os, sys, time, random, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
torch.set_num_threads(max(1, (os.cpu_count() or 2) // 2))
torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Locate the repo: works whether the notebook sits next to WeakTr/ or inside it.
ROOT = Path.cwd()
REPO = ROOT / "WeakTr" if (ROOT / "WeakTr" / "models.py").exists() else ROOT
assert (REPO / "models.py").exists(), "Open this notebook from the repository root or its parent"
CACHE = Path(os.environ.get("WEAKTR_CACHE", str(REPO.parent / "weaktr_cache")))
CACHE.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| device:", DEVICE, "| repo:", REPO)

In [ ]:
import requests

def download(url, dest, desc=""):
    # Download with resume support; skip if the file is already complete.
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0 and not dest.with_suffix(dest.suffix + ".part").exists():
        return dest
    part = dest.with_suffix(dest.suffix + ".part")
    done = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={done}-"} if done else {}
    print(f"downloading {desc or dest.name} ...", end=" ", flush=True)
    with requests.get(url, stream=True, headers=headers, timeout=60) as r:
        r.raise_for_status()
        mode = "ab" if (done and r.status_code == 206) else "wb"
        with open(part, mode) as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    part.replace(dest)
    print(f"{dest.stat().st_size / 1e6:.1f} MB")
    return dest

def gdrive(file_id):
    return f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"

# 1) Two example images (COCO val2017; they contain PASCAL VOC classes)
IMG_CATS = download("http://images.cocodataset.org/val2017/000000039769.jpg", CACHE / "cats.jpg")
IMG_DOG  = download("http://images.cocodataset.org/val2017/000000017029.jpg", CACHE / "dog_car.jpg")
# 2) DeiT-Small ImageNet weights: the starting point of Phase 1 (main.py --finetune default)
DEIT_CKPT = download("https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth",
                     CACHE / "deit_small_patch16_224.pth", "DeiT-S ImageNet weights (88 MB)")
# 3) Official WeakTr Phase-1 checkpoint for VOC12 (README, "Step1" table)
WEAKTR_CKPT = download(gdrive("1uUHGLIDFm49Gh41Ddc8_T93-bRWnJzeF"), CACHE / "weaktr_voc_phase1.pth",
                       "official WeakTr Phase-1 checkpoint (87 MB)")

In [ ]:
# PASCAL VOC: 20 object classes (+ "background" = index 0 in segmentation masks)
VOC_CLASSES = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
               'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa',
               'train', 'tvmonitor']
MEAN = torch.tensor([0.485, 0.456, 0.406])[:, None, None]  # ImageNet statistics (DeiT normalisation)
STD = torch.tensor([0.229, 0.224, 0.225])[:, None, None]

def load_image(path):
    return Image.open(path).convert("RGB")

def to_tensor(pil):
    # PIL image -> normalised (1, 3, H, W) tensor, exactly as datasets.py build_transform does at CAM time
    x = torch.from_numpy(np.array(pil)).permute(2, 0, 1).float() / 255.
    return ((x - MEAN) / STD)[None]

def overlay(ax, img, heat, title="", alpha=0.5, cmap="jet"):
    ax.imshow(img); ax.imshow(heat, cmap=cmap, alpha=alpha,
                              extent=(0, img.size[0], img.size[1], 0) if hasattr(img, "size") else None)
    ax.set_title(title, fontsize=9); ax.axis("off")

def voc_cmap(n=256):
    # the standard PASCAL VOC colour palette
    cmap = np.zeros((n, 3), np.uint8)
    for i in range(n):
        r = g = b = 0; c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j); g |= ((c >> 1) & 1) << (7 - j); b |= ((c >> 2) & 1) << (7 - j); c >>= 3
        cmap[i] = (r, g, b)
    return cmap
CMAP = voc_cmap()

img_cats, img_dog = load_image(IMG_CATS), load_image(IMG_DOG)
LABELS = {"cats": ["cat", "sofa"], "dog": ["dog", "car"]}  # the *only* supervision WSSS gets

---
# Part 1: The problem WeakTr solves

### Semantic segmentation
**Semantic segmentation** means giving every pixel a class name ("this pixel is cat, that one is sofa, that one is background"). A normal (*fully-supervised*) segmentation network is trained on images where a human has painted every pixel. That painting is slow and expensive, often several minutes per image.

### Weakly-supervised semantic segmentation (WSSS)
WSSS asks a cheaper question: **can we learn to segment from image-level labels only?** These are labels like "this image contains a cat and a sofa", with no information about *where*.

Here is exactly what WeakTr gets to see for our two example images:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, img, key in zip(axs, [img_cats, img_dog], ["cats", "dog"]):
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"image-level label = {LABELS[key]}\n(no pixel annotation at all)", fontsize=10)
plt.tight_layout(); plt.show()

# How the repo stores these labels: a 20-dim multi-hot vector per image (voc12/cls_labels.npy)
cls_labels = np.load(REPO / "voc12" / "cls_labels.npy", allow_pickle=True).item()
name, vec = next(iter(cls_labels.items()))
print("example entry of voc12/cls_labels.npy:", name, "->", vec.astype(int),
      "=", [VOC_CLASSES[i] for i in np.nonzero(vec)[0]])

### The general WSSS recipe, and where WeakTr changes it

1. **Train a classifier** on the image-level labels.
2. Ask the classifier **"where did you look to decide 'cat'?"** The answer is a heat map called a **Class Activation Map (CAM)**.
3. Turn the CAMs into **pseudo masks** (fake pixel labels): high CAM means object, low CAM means background.
4. **Train a real segmentation network** on these noisy pseudo masks.

Earlier transformer methods (e.g. MCTformer) did this in **three** phases: CAM generation, then a slow CAM refinement network (AffinityNet / random walk), then retraining. WeakTr uses **two** phases (paper Fig. A8):

| | WeakTr phase | Key new idea | Paper section |
|---|---|---|---|
| Phase 1 | **End-to-end CAM generation** | **Adaptive Attention Fusion (AAF)**: learn *how much to trust each attention head* instead of simply averaging them | III-A |
| Phase 2 | **Online retraining** | **Gradient-clipping decoder**: during training, ignore the pixels whose loss is unusually large, because those are probably wrong pseudo labels | III-B |

The authors' figures of the two phases ship with the repo:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [2.2, 1]})
for ax, f, t in zip(axs, ["WeakTr.png", "clip_grad_decoder.png"],
                    ["Phase 1: end-to-end CAM generation (paper Fig. 2)", "Phase 2: gradient clipping decoder (paper Fig. 4)"]):
    p = REPO / "img" / f
    if p.exists():
        ax.imshow(Image.open(p)); ax.set_title(t)
    ax.axis("off")
plt.tight_layout(); plt.show()

Don't worry if the figures look dense right now. By the end of Part 3 you'll have built every box in the left figure, and by the end of Part 5 every box in the right one.

---
# Part 2: Vision Transformer (ViT) crash course

WeakTr uses a **plain ViT**, specifically **DeiT-Small** (`deit_small_patch16_224`): 12 layers, 6 attention heads per layer, 384 numbers per token. Let's take it apart.

## 2.1 Images become a sequence of patch tokens
A ViT cuts the image into non-overlapping **16×16 patches**. A 224×224 image gives a 14×14 grid, so $N^2 = 196$ patches. Each patch is flattened and linearly mapped to a vector of length $D=384$, called a **token**. In code, this "flatten + linear" is one strided convolution (`PatchEmbed` in `vision_transformer.py`, L127-144).

In [ ]:
crop = img_cats.resize((224, 224))
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
axs[0].imshow(crop)
for k in range(0, 225, 16):
    axs[0].axhline(k - .5, color="w", lw=.6); axs[0].axvline(k - .5, color="w", lw=.6)
axs[0].set_title("224×224 image cut into 14×14 = 196 patches of 16×16"); axs[0].axis("off")

patch_embed = nn.Conv2d(3, 384, kernel_size=16, stride=16)   # = PatchEmbed.proj
tokens = patch_embed(to_tensor(crop)).flatten(2).transpose(1, 2)
print("image tensor:", tuple(to_tensor(crop).shape), "->  patch tokens:", tuple(tokens.shape), "(batch, N², D)")

# show a few individual patches
for i, (r, c) in enumerate([(3, 5), (6, 3), (10, 10)]):
    a = fig.add_axes([0.56 + 0.14 * i, 0.3, 0.12, 0.4])
    a.imshow(np.array(crop)[r*16:(r+1)*16, c*16:(c+1)*16]); a.set_title(f"patch #{r*14+c}", fontsize=8); a.axis("off")
axs[1].axis("off"); axs[1].set_title("each patch becomes one 384-dim token")
plt.show()

## 2.2 Self-attention: how tokens talk to each other (paper Eq. 1–2)

Every token produces three vectors: a **query** $Q$ ("what am I looking for?"), a **key** $K$ ("what do I contain?") and a **value** $V$ ("what will I pass on?").

$$A = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt d}\right) \qquad\qquad AF = A\,V$$

* $A$ is the **attention map**, a square matrix. Row $i$ says how much token $i$ looks at every other token, and each row sums to 1.
* $AF$ is the **attention feature**: each token's output is a weighted average of the values.

Here is a toy example with **3 tokens** and dimension $d=2$, so you can follow every number:

In [ ]:
torch.manual_seed(1)
X = torch.tensor([[1.0, 0.0],    # token 0
                  [0.9, 0.1],    # token 1 (similar to token 0)
                  [0.0, 1.0]])   # token 2 (different)
Wq, Wk, Wv = torch.eye(2), torch.eye(2), torch.tensor([[1., 2.], [3., 4.]])
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / math.sqrt(2)
A = scores.softmax(dim=-1)
np.set_printoptions(precision=3, suppress=True)
print("Q·Kᵀ/√d =\n", scores.numpy())
print("\nA = softmax(...) (each row sums to 1) =\n", A.numpy())
print("\nAF = A·V =\n", (A @ V).numpy())
print("\n-> token 0 attends mostly to itself and to token 1 (they are similar), less to token 2.")

**Multi-head attention**: each layer runs this $H=6$ times in parallel. Each run is a **head** with its own $W_Q, W_K, W_V$, so each head can learn to look for something different. With $K=12$ layers that's **$K \times H = 72$ attention maps** per image.

> **This is WeakTr's starting observation (paper Fig. 1 and Fig. 7):** different heads focus on different things. Some follow the object, some the background or context, some just a fixed screen position. Averaging all 72 maps equally, as earlier methods did, mixes useful heads with misleading ones. Part 3 shows this on our image.

---
# Part 3: Phase 1, end-to-end CAM generation

We now build the WeakTr Phase-1 network, following paper section III-A and `models.py` / `AAF.py` / `vision_transformer.py`.

## 3.1 The building blocks (copied from `vision_transformer.py`)
The standard ViT block with one change: it **returns the attention map** $A$ and the **attention output** $AF$ as well, because WeakTr needs them (`Block.forward` returns `x, weights, o`).

Each block is: LayerNorm, multi-head self-attention, a residual connection, then LayerNorm, an MLP, and another residual connection.

In [ ]:
class DropPath(nn.Module):                         # stochastic depth (timm.models.layers.DropPath)
    def __init__(self, p=0.):
        super().__init__(); self.p = p
    def forward(self, x):
        if self.p == 0. or not self.training: return x
        keep = x.new_empty(x.shape[0], *([1] * (x.ndim - 1))).bernoulli_(1 - self.p)
        return x * keep / (1 - self.p)

class Mlp(nn.Module):                              # vision_transformer.py L54-70
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Attention(nn.Module):                        # vision_transformer.py L73-103
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5    # 1/sqrt(d)
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = ((q @ k.transpose(-2, -1)) * self.scale).softmax(dim=-1)   # Eq. (1): B x heads x N x N
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)                   # Eq. (2)
        return self.proj(x), attn

class Block(nn.Module):                            # vision_transformer.py L106-124
    def __init__(self, dim, num_heads, drop_path=0.):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim, eps=1e-6), Attention(dim, num_heads)
        self.norm2, self.mlp = nn.LayerNorm(dim, eps=1e-6), Mlp(dim, dim * 4)
        self.drop_path = DropPath(drop_path)
    def forward(self, x):
        o, weights = self.attn(self.norm1(x))
        x = x + self.drop_path(o)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x, weights, o          # <- WeakTr also returns the attention map and attention output

## 3.2 Adaptive Attention Fusion, the core idea of Phase 1 (paper Eq. 3–5, `AAF.py`)

Goal: produce **one weight per head** ($72$ weights) that says how useful each attention map is for building the CAM.

1. **Pooling** (Eq. 3): for each head, take its attention feature $AF_i$ (shape: tokens × 64) and **max-pool over the tokens**, giving a 64-dim summary of that head.
2. **Head-wise FFN** (Eq. 3, `attn_feat_ffn`): a tiny network $64 \to 8 \to 1$ applied to every head separately turns each summary into one number. Result: the **dynamic weights** $W$, shape 72.
3. **Fully-connected FFN** (Eq. 4, `attn_head_ffn`): $72 \to 9 \to 72$ + **sigmoid** lets the heads "talk" to each other, giving the **interacted weights** $W' \in (0,1)^{72}$. (Table V: a hidden size of 9 works best.)
4. **Weighted fusion** (Eq. 5): $\hat A = \frac{1}{KH}\sum_i W'_i\,A_i$. Each attention map is multiplied by its weight, then all are averaged.

Because the weights depend on the input image ("dynamic"), a head can be trusted for one image and not for another.

In [ ]:
class AAF(nn.Module):                              # AAF.py L6-43 (repo default: reduction=8, pool="max")
    def __init__(self, channel=72, reduction=8, feats_channel=64, feat_reduction=8, pool="max"):
        super().__init__()
        self.avg_pool = nn.AdaptiveMaxPool2d(1) if pool == "max" else nn.AdaptiveAvgPool2d(1)
        self.attn_head_ffn = nn.Sequential(                       # Eq. (4): fully-connected FFN across heads
            nn.Linear(channel, int(channel / reduction), bias=False), nn.ReLU(inplace=True),
            nn.Linear(int(channel / reduction), channel, bias=False), nn.Sigmoid())
        self.attn_feat_ffn = nn.Sequential(                       # Eq. (3): head-wise FFN
            nn.Linear(feats_channel, int(feats_channel / feat_reduction)),
            nn.Linear(int(feats_channel / feat_reduction), 1))

    def forward_weight(self, x):
        b, c, n, m = x.size()            # batch, 72 heads, tokens, 64 features per head
        x = x.permute(0, 1, 3, 2).contiguous().view(b, c * m, n, 1)
        attn_feat_pool = self.avg_pool(x)                          # 1. pool over the tokens
        attn_weight = self.attn_feat_ffn(attn_feat_pool.view(b * c, m))   # 2. one number per head
        attn_weight = self.attn_head_ffn(attn_weight.view(b, c))   # 3. heads interact, sigmoid
        return attn_weight.view(b, c, -1, 1)

    def forward(self, attn_feat, x):
        weight = self.forward_weight(attn_feat)
        return x * weight.expand_as(x), x * weight.expand_as(x)    # weighted cross- and patch-attention

## 3.3 The full WeakTr Phase-1 model (`models.py`)

Two more ideas complete the model.

**Multiple class tokens** (borrowed from MCTformer): a normal ViT adds one learnable `[CLS]` token. WeakTr adds **one class token per class, so $C=20$**. The input sequence is therefore `[20 class tokens | N² patch tokens]`. The attention of the *cat* class token to the patches then says *where the cat is*.

Each $(20+N^2)\times(20+N^2)$ attention map splits into blocks:

```
              keys → | 20 class tokens |   N² patch tokens   |
 queries ↓  ---------+-----------------+---------------------+
 20 class tokens     |   (unused)      |  CROSS-attention CA  |  "where does class c look?"   (20 × N²)
 N² patch tokens     |   (unused)      |  PATCH-attention PA  |  "which patches are similar?" (N² × N²)
```

**Three ways to predict the classes** (paper Fig. 2, Eq. 6–9):

| Output | How | Loss |
|---|---|---|
| $\hat P_{CLS}$ | mean of each class token's 384 features | $L_{CLS-token}$ |
| **Coarse CAM** | patch tokens arranged as an $N\times N$ image, then a 3×3 conv with 20 output channels (Eq. 6–7); global average pooling gives $\hat P_{coarse}$ | $L_{Coarse-CAM}$ |
| **Fine CAM** | $\widehat{PA}\cdot(\text{CAM}_{coarse}\odot\widehat{CA})$ (Eq. 8), using the **AAF-weighted** attention; pooling gives $\hat P_{fine}$ | $L_{Fine-CAM}$ |

The code below matches `models.py` `WeakTr.forward` line for line. The comments give the paper equations.

In [ ]:
class WeakTr(nn.Module):                           # models.py L19-144
    def __init__(self, num_classes=20, embed_dim=384, depth=12, num_heads=6, patch_size=16,
                 img_size=224, drop_path_rate=0.1):
        super().__init__()
        self.num_classes, self.embed_dim, self.patch_size = num_classes, embed_dim, patch_size
        self.patch_embed = nn.Module()
        self.patch_embed.proj = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, num_classes, embed_dim))          # C class tokens
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + num_classes, embed_dim))
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, dpr[i]) for i in range(depth)])
        self.head = nn.Conv2d(embed_dim, num_classes, kernel_size=3, stride=1, padding=1)  # coarse-CAM conv
        self.adaptive_attention_fusion = AAF(channel=depth * num_heads, reduction=8, pool="max")
        nn.init.trunc_normal_(self.cls_token, std=.02); nn.init.trunc_normal_(self.pos_embed, std=.02)

    def interpolate_pos_encoding(self, x, w, h):   # lets the model accept any image size
        npatch, N = x.shape[1] - self.num_classes, self.pos_embed.shape[1] - self.num_classes
        if npatch == N and w == h:
            return self.pos_embed
        class_pos_embed = self.pos_embed[:, :self.num_classes]
        patch_pos_embed = self.pos_embed[:, self.num_classes:]
        dim = x.shape[-1]
        w0, h0 = w // self.patch_size + 0.1, h // self.patch_size + 0.1
        patch_pos_embed = F.interpolate(
            patch_pos_embed.reshape(1, int(math.sqrt(N)), int(math.sqrt(N)), dim).permute(0, 3, 1, 2),
            scale_factor=(w0 / math.sqrt(N), h0 / math.sqrt(N)), mode="bicubic")
        return torch.cat((class_pos_embed, patch_pos_embed.permute(0, 2, 3, 1).reshape(1, -1, dim)), dim=1)

    def forward_features(self, x):
        B, nc, w, h = x.shape
        x = self.patch_embed.proj(x).flatten(2).transpose(1, 2)                 # patch tokens
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1)             # T0 = [class | patch]
        x = x + self.interpolate_pos_encoding(x, w, h)
        attn_weights, attn_feats = [], []
        for blk in self.blocks:
            x, weights_i, feat = blk(x)
            attn_weights.append(weights_i); attn_feats.append(feat)
        return x[:, :self.num_classes], x[:, self.num_classes:], attn_weights, attn_feats

    def forward(self, x, return_att=False, attention_type="fused", return_all=False):
        w, h = x.shape[2:]
        x_cls, x_patch, attn_weights, attn_feats = self.forward_features(x)
        n, p, c = x_patch.shape
        w0, h0 = w // self.patch_size, h // self.patch_size
        x_patch = x_patch.reshape(n, w0, h0, c).permute(0, 3, 1, 2).contiguous()  # Eq. (6) Arrange
        x_patch = self.head(x_patch)                                             # Eq. (7) coarse CAM
        coarse_cam_pred = x_patch.mean((2, 3))                                   # avg pooling -> P_coarse

        # Stack all K*H = 72 attention maps / features. NOTE .detach(): see section 3.6
        attn_weights = torch.stack(attn_weights).detach()                        # K, B, H, N, N
        k, b, hh, nn_, m = attn_weights.shape
        attn_weights = attn_weights.permute(1, 2, 0, 3, 4).reshape(b, hh * k, nn_, m)
        attn_feats = torch.stack(attn_feats).detach()                            # K, B, N, C
        attn_feats = attn_feats.view(k, b, nn_, -1, hh).permute(1, 4, 0, 2, 3).reshape(b, hh * k, nn_, -1)
        cross_attn_map, patch_attn_map = self.adaptive_attention_fusion(attn_feats, attn_weights)  # Eq. (3-4)

        coarse_cam = F.relu(x_patch.detach().clone())
        C = self.num_classes
        cross_attn = cross_attn_map.mean(1)[:, :C, C:].reshape(n, C, w0, h0)     # Eq. (5) -> weighted CA
        if attention_type == "fused":
            cams = cross_attn * coarse_cam                                       # CAM_coarse ⊙ CA
        elif attention_type == "patchcam":
            cams = coarse_cam
        else:
            cams = cross_attn
        patch_attn = patch_attn_map.mean(1)[:, C:, C:]                           # weighted PA (N² x N²)
        fine_cam = torch.matmul(patch_attn.unsqueeze(1), cams.view(n, C, -1, 1)).reshape(n, C, w0, h0)  # Eq. (8)
        fine_cam_pred = fine_cam.mean((2, 3))
        cls_token_pred = x_cls.mean(-1)

        if return_all:   # extra hook for this notebook: expose every intermediate result
            return dict(cls_pred=cls_token_pred, coarse_pred=coarse_cam_pred, fine_pred=fine_cam_pred,
                        coarse_cam=coarse_cam, cross_attn=cross_attn, cams=cams, patch_attn=patch_attn,
                        fine_cam=fine_cam, attn_all=attn_weights,
                        head_weights=self.adaptive_attention_fusion.forward_weight(attn_feats)[:, :, 0, 0],
                        grid=(w0, h0))
        if return_att:
            return cls_token_pred, cams, patch_attn.unsqueeze(0)
        return cls_token_pred, coarse_cam_pred, fine_cam_pred

## 3.4 How the model is initialised: starting from ImageNet DeiT-S (`main.py` L301-349)

WeakTr doesn't start from scratch. It loads **DeiT-Small trained on ImageNet** and adapts it:

* DeiT has **1** class token, but WeakTr needs **20**, so the pretrained token (and its position embedding) is **copied 20 times**.
* The ImageNet classification head is dropped, because the new `head` is a conv layer.
* The AAF module and the conv head start random.

In [ ]:
def build_weaktr_from_deit(deit_path, num_classes=20):
    model = WeakTr(num_classes=num_classes)
    ckpt = torch.load(deit_path, map_location="cpu")["model"]
    sd = model.state_dict()
    for k in ["head.weight", "head.bias", "head_dist.weight", "head_dist.bias"]:
        if k in ckpt and (k not in sd or ckpt[k].shape != sd[k].shape):
            del ckpt[k]
    pos = ckpt["pos_embed"]                                   # (1, 1 + 196, 384)
    ckpt["pos_embed"] = torch.cat([pos[:, :1].repeat(1, num_classes, 1), pos[:, 1:]], dim=1)
    ckpt["cls_token"] = ckpt["cls_token"].repeat(1, num_classes, 1)
    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    return model, missing, unexpected

model_init, missing, unexpected = build_weaktr_from_deit(DEIT_CKPT)
print("cls_token :", tuple(model_init.cls_token.shape), "  pos_embed:", tuple(model_init.pos_embed.shape))
print("randomly initialised (not in DeiT):", sorted({k.split('.')[0] for k in missing}))
print("ignored DeiT weights:", unexpected)
print(f"parameters: {sum(p.numel() for p in model_init.parameters()) / 1e6:.1f} M")

## 3.5 A forward pass, shape by shape
Let's push one 224×224 image through the freshly initialised model and print the size of everything. This is the best way to really *see* the architecture.

In [ ]:
x = to_tensor(img_cats.resize((224, 224)))
with torch.no_grad():
    out = model_init.eval()(x, return_all=True)
rows = [("input image", x), ("all 72 attention maps A*", out["attn_all"]),
        ("coarse CAM  (Eq. 7)", out["coarse_cam"]), ("AAF head weights W' (Eq. 4)", out["head_weights"]),
        ("weighted cross-attention CA^", out["cross_attn"]), ("weighted patch-attention PA^", out["patch_attn"]),
        ("fine CAM (Eq. 8)", out["fine_cam"]),
        ("P_cls / P_coarse / P_fine", out["cls_pred"])]
for name, t in rows:
    print(f"{name:32s} {str(tuple(t.shape)):>22s}")
print("\n216 = 20 class tokens + 196 patch tokens;  72 = 12 layers x 6 heads;  14x14 = patch grid")

## 3.6 Training: three losses, and a subtle detail about gradients

All three predictions are compared with the image-level label $y\in\{0,1\}^{20}$ using the **multi-label soft-margin loss**, i.e. a binary cross-entropy per class:

$$\mathrm{Loss}(\hat P, y) = -\frac1C\sum_{i=1}^{C}\Big[y_i\log\sigma(\hat P_i) + (1-y_i)\log\big(1-\sigma(\hat P_i)\big)\Big],\qquad L = L_{CLS} + L_{coarse} + L_{fine}\quad(\text{Eq. 9})$$

Let's check that formula against PyTorch's built-in loss, which is what `engine.py` uses:

In [ ]:
P_hat = torch.tensor([[2.0, -1.0, 0.5]])   # scores for 3 classes
y = torch.tensor([[1.0, 0.0, 1.0]])        # classes 0 and 2 are present
by_hand = -(y * torch.log(torch.sigmoid(P_hat)) + (1 - y) * torch.log(1 - torch.sigmoid(P_hat))).mean()
print("by hand :", by_hand.item())
print("PyTorch :", F.multilabel_soft_margin_loss(P_hat, y).item())

**Who learns from which loss?** In `models.py` the attention maps, the attention features *and* the coarse CAM are passed through `.detach()` before the AAF / fine-CAM branch. So:

* $L_{CLS}$ and $L_{coarse}$ train the **ViT backbone** and the **conv head**.
* $L_{fine}$ trains **only the AAF module**. It learns *which heads to trust* without disturbing the backbone.

That is what the paper means by AAF being trained **end-to-end with image-level supervision** (Sec. III-A-4). We can check it directly:

In [ ]:
def who_gets_gradient(model, loss):
    model.zero_grad(); loss.backward(retain_graph=True)
    groups = {}
    for n_, p in model.named_parameters():
        g = n_.split(".")[0]
        groups.setdefault(g, False)
        groups[g] |= (p.grad is not None and p.grad.abs().sum().item() > 0)
    return [g for g, v in groups.items() if v]

model_init.train()
y = torch.zeros(1, 20); y[0, VOC_CLASSES.index("cat")] = 1; y[0, VOC_CLASSES.index("sofa")] = 1
p_cls, p_coarse, p_fine = model_init(x)
for name, l in [("L_CLS-token", F.multilabel_soft_margin_loss(p_cls, y)),
                ("L_coarse-CAM", F.multilabel_soft_margin_loss(p_coarse, y)),
                ("L_fine-CAM", F.multilabel_soft_margin_loss(p_fine, y))]:
    print(f"{name:13s} = {l.item():.3f}  -> updates: {who_gets_gradient(model_init, l)}")

### A miniature training run
The real recipe (`docs/Training.md`) trains for 60 epochs on 10,582 VOC images with batch size 64, which takes about 2h40 on a GPU (paper Table XI). Here we only run **a handful of steps** on our two images with random crops, so you can watch the training loop of `engine.py` `train_one_epoch` at work. Don't expect good CAMs from this; in the next section we load the authors' fully trained weights.

In [ ]:
import torchvision.transforms as T
aug = T.Compose([T.RandomResizedCrop(224, scale=(0.3, 1.0)), T.RandomHorizontalFlip(),
                 T.ToTensor(), T.Normalize(MEAN.flatten().tolist(), STD.flatten().tolist())])
def multi_hot(names):
    v = torch.zeros(20)
    for n_ in names: v[VOC_CLASSES.index(n_)] = 1
    return v
data = [(img_cats, multi_hot(LABELS["cats"])), (img_dog, multi_hot(LABELS["dog"]))]

model_toy = build_weaktr_from_deit(DEIT_CKPT)[0].train()
opt = torch.optim.AdamW(model_toy.parameters(), lr=6e-4 * 4 / 512, weight_decay=0.03)  # main.py: lr scaled by batch/512
hist = []
t0 = time.time()
for step in range(12):
    batch = [data[i % 2] for i in range(4)]
    xb = torch.stack([aug(im) for im, _ in batch]); yb = torch.stack([l for _, l in batch])
    p_cls, p_coarse, p_fine = model_toy(xb)
    losses = [F.multilabel_soft_margin_loss(p, yb) for p in (p_cls, p_coarse, p_fine)]
    loss = sum(losses)                                       # Eq. (9)
    opt.zero_grad(); loss.backward(); opt.step()
    hist.append([l.item() for l in losses])
print(f"{len(hist)} steps in {time.time() - t0:.0f}s")
hist = np.array(hist)
plt.figure(figsize=(6, 3))
for i, n_ in enumerate(["L_CLS-token", "L_coarse-CAM", "L_fine-CAM"]): plt.plot(hist[:, i], label=n_)
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("toy training (2 images)"); plt.show()
del model_toy, opt

## 3.7 Load the authors' trained Phase-1 model
This is the official VOC12 checkpoint from the README's "Step1" table. It was trained for the full recipe and selected by CAM quality.

In [ ]:
ckpt = torch.load(WEAKTR_CKPT, map_location="cpu")
weaktr = WeakTr(num_classes=20)
missing, unexpected = weaktr.load_state_dict(ckpt["model"], strict=False)
weaktr.eval().to(DEVICE)
print("missing weights   :", missing)
print("unused weights    :", unexpected, "(the final LayerNorm of DeiT is never used by WeakTr)")
print("checkpoint epoch  :", ckpt["epoch"], "| train-set CAM mIoU recorded by the authors: %.1f%%" % ckpt["miou"])

@torch.no_grad()
def run(model, pil):
    return model(to_tensor(pil).to(DEVICE), return_all=True)

for key, img in [("cats", img_cats), ("dog", img_dog)]:
    o = run(weaktr, img)
    for name in ["cls_pred", "coarse_pred"]:
        top = torch.sigmoid(o[name][0]).topk(3)
        print(f"{key:5s} {name:12s} top-3:", [(VOC_CLASSES[i], round(v.item(), 2)) for v, i in zip(top.values, top.indices)])

The classifier recognises the classes. Note that *sofa* isn't confidently detected in the cat image, because the sofa is mostly covered by a pink blanket. That doesn't matter for CAM generation, because at that stage the **ground-truth image labels** are used to pick which class maps to keep.

## 3.8 Look inside: what the 72 heads see, and how AAF weights them (paper Fig. 6, 7)

For the **cat** class token, we plot its cross-attention to the patches in each of the 72 heads (12 layers × 6 heads). The number above each map is the **AAF weight $W'$** for that head on this image.

In [ ]:
def head_grid(model, pil, cls_name, layers=range(12)):
    o = run(model, pil)
    c = VOC_CLASSES.index(cls_name); C = 20
    w0, h0 = o["grid"]
    A = o["attn_all"][0].cpu()            # (72, N, N), ordered head-major: index = head*12 + layer
    W = o["head_weights"][0].cpu()        # (72,)
    fig, axs = plt.subplots(6, len(layers), figsize=(1.45 * len(layers), 6.2))
    for hd in range(6):
        for j, L in enumerate(layers):
            idx = hd * 12 + L
            ax = axs[hd, j]
            ax.imshow(A[idx, c, C:].reshape(w0, h0), cmap="jet"); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"W'={W[idx]:.2f}", fontsize=7, color="green" if W[idx] > W.median() else "gray")
            if j == 0: ax.set_ylabel(f"head {hd}", fontsize=8)
            if hd == 5: ax.set_xlabel(f"layer {L}", fontsize=8)
    fig.suptitle(f"Cross-attention of the '{cls_name}' class token in every head (green = AAF weight above median)", fontsize=11)
    plt.tight_layout(); plt.show()
    return o

o_cats = head_grid(weaktr, img_cats, "cat")

Look for:
* **early layers** (left), whose attention is spread out and fairly uninformative;
* heads that light up the **cats**;
* heads that light up the **remotes, the blanket, or a fixed spot**.

Averaging all of these equally ("mean-sum", as in MCTformer) lets the bad heads blur the result. AAF learns to **down-weight** them.

Below are the AAF weights as a 12×6 table, plus the effect of the weighting: the same fusion with **all weights set to 1** (the mean-sum baseline) next to the **AAF-weighted** version. This reproduces Fig. 6 (b) vs (c) and (e) vs (f).

In [ ]:
def fuse(o, weights, cls_name):
    # Eq. (5) with arbitrary per-head weights -> (cross-attention for one class, patch attention)
    A = o["attn_all"][0].cpu(); C = 20; c = VOC_CLASSES.index(cls_name); w0, h0 = o["grid"]
    Ahat = (A * weights[:, None, None]).mean(0)
    return Ahat[c, C:].reshape(w0, h0), Ahat[C:, C:]

W = o_cats["head_weights"][0].cpu()
fig, axs = plt.subplots(1, 5, figsize=(19, 3.6), gridspec_kw={"width_ratios": [1.1, 1, 1, 1, 1]})
im = axs[0].imshow(W.reshape(6, 12), cmap="viridis"); plt.colorbar(im, ax=axs[0], fraction=.03)
axs[0].set_xlabel("layer"); axs[0].set_ylabel("head"); axs[0].set_title("AAF weights W' (cats image)")
ca_mean, pa_mean = fuse(o_cats, torch.ones(72), "cat")
ca_aaf, pa_aaf = fuse(o_cats, W, "cat")
up = lambda m: np.array(Image.fromarray(m.numpy().astype(np.float32)).resize(img_cats.size, Image.BILINEAR))
overlay(axs[1], img_cats, up(ca_mean), "cross-attention 'cat': mean-sum")
overlay(axs[2], img_cats, up(ca_aaf), "cross-attention 'cat': AAF-weighted")
w0, h0 = o_cats["grid"]; q = 2 * h0 + 2          # a background query patch (top-left area, sofa)
overlay(axs[3], img_cats, up(pa_mean[q].reshape(w0, h0)), "patch-attention from ★: mean-sum")
overlay(axs[4], img_cats, up(pa_aaf[q].reshape(w0, h0)), "patch-attention from ★: AAF-weighted")
for ax in axs[3:]: ax.plot(2.5 * 16 * img_cats.size[0] / (h0 * 16), 2.5 * 16 * img_cats.size[1] / (w0 * 16), "w*", ms=14)
plt.tight_layout(); plt.show()
print(f"weights range {W.min():.2f} .. {W.max():.2f}; mean {W.mean():.2f}.  "
      f"Heads with W' < 0.3: {(W < 0.3).sum().item()} of 72")

## 3.9 From coarse CAM to fine CAM (Eq. 6–8), visualised

For the class *cat*:
1. **coarse CAM**: what the conv head on the patch tokens says. It is usually blobby and focused on the most distinctive parts.
2. **weighted cross-attention $\widehat{CA}$**: where the *cat* class token looks.
3. **fused** = coarse ⊙ CA: both must agree.
4. **fine CAM** = $\widehat{PA}\cdot$fused: each patch collects the scores of the patches *similar to it*, which spreads activation over the whole object.

In [ ]:
def show_cam_steps(o, img, cls_name):
    c = VOC_CLASSES.index(cls_name)
    norm = lambda m: (m - m.min()) / (m.max() - m.min() + 1e-8)
    up = lambda m: np.array(Image.fromarray(norm(m).cpu().numpy().astype(np.float32)).resize(img.size, Image.BILINEAR))
    steps = [("coarse CAM (Eq. 7)", o["coarse_cam"][0, c]), ("weighted cross-attn CA^", o["cross_attn"][0, c]),
             ("CAM_coarse ⊙ CA^", o["cams"][0, c]), ("fine CAM = PA^ · (...)  (Eq. 8)", o["fine_cam"][0, c])]
    fig, axs = plt.subplots(1, 4, figsize=(18, 3.6))
    for ax, (t, m) in zip(axs, steps): overlay(ax, img, up(m), f"{cls_name}: {t}")
    plt.tight_layout(); plt.show()

show_cam_steps(o_cats, img_cats, "cat")
o_dog = run(weaktr, img_dog)
show_cam_steps(o_dog, img_dog, "dog")
show_cam_steps(o_dog, img_dog, "car")

## 3.10 How CAMs are actually produced for the dataset (`engine.py` `generate_attention_maps_ms`)

At CAM-generation time, the repo:
1. runs the image at **several scales** (VOC: 1.0 and 1.2) **and their horizontal flips**;
2. computes the fine CAM for each (the `patch_attn_refine` step is Eq. 8 again);
3. upsamples each map to the original resolution, **keeps only the classes in the ground-truth image label** (multiplies by `target`), un-flips, and sums;
4. **min-max normalises** each class map to [0, 1] and saves a dictionary `{class_index: cam}` as a `.npy` file.

In [ ]:
@torch.no_grad()
def generate_cam(model, pil, label_names, scales=(1.0, 1.2), patch=16):
    target = multi_hot(label_names).to(DEVICE)
    W0, H0 = pil.size
    cam_list = []
    for s in scales:
        im = pil.resize((round(W0 * s), round(H0 * s)), Image.BICUBIC)   # datasets.py VOC12DatasetMS
        x = to_tensor(im).to(DEVICE)
        for flip in (False, True):
            xi = torch.flip(x, [-1]) if flip else x
            w_feat, h_feat = xi.shape[2] // patch, xi.shape[3] // patch
            _, cams, patch_attn = model(xi, return_att=True, attention_type="fused")
            patch_attn = torch.sum(patch_attn, dim=0)
            cams = torch.matmul(patch_attn.unsqueeze(1), cams.view(cams.shape[0], cams.shape[1], -1, 1)) \
                        .reshape(cams.shape[0], cams.shape[1], w_feat, h_feat)            # patch_attn_refine
            cams = F.interpolate(cams, size=(H0, W0), mode="bilinear", align_corners=False)[0]
            cams = cams * target.view(20, 1, 1)                                           # keep GT classes only
            if flip: cams = torch.flip(cams, dims=[-1])
            cam_list.append(cams)
    sum_cam = torch.stack(cam_list).sum(0)
    cam_dict = {}
    for c in torch.nonzero(target).flatten().tolist():
        cam = sum_cam[c]
        cam_dict[c] = ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy()   # min-max normalise
    return cam_dict

t0 = time.time()
cam_cats = generate_cam(weaktr, img_cats, LABELS["cats"])
cam_dog = generate_cam(weaktr, img_dog, LABELS["dog"])
print(f"multi-scale + flip CAMs for 2 images in {time.time() - t0:.1f}s; saved format:",
      {k: (VOC_CLASSES[k], v.shape) for k, v in cam_cats.items()})
fig, axs = plt.subplots(1, 4, figsize=(18, 3.6))
for ax, (img, d, c) in zip(axs, [(img_cats, cam_cats, k) for k in cam_cats] + [(img_dog, cam_dog, k) for k in cam_dog]):
    overlay(ax, img, d[c], f"final CAM: {VOC_CLASSES[c]}")
plt.tight_layout(); plt.show()

---
# Part 4: From CAM to pseudo mask, and how quality is measured

## 4.1 The background threshold $t$ (`evaluation.py` `do_python_eval`)
A CAM says how "cat-like" each pixel is, but nothing says *background*. The repo invents a background channel with a **constant score $t$**. Then for every pixel it picks the class with the highest score (`argmax` after `softmax`). So a pixel becomes background if all class CAMs are below $t$.

The best $t$ is found by a sweep (`--curve True --start 40`). For VOC the authors use **$t=0.43$** (`--t 43` in `docs/Training.md`).

In [ ]:
def cam_to_mask(cam_dict, t):
    # evaluation.py L120-139 (without CRF)
    keys = list(cam_dict.keys())
    cam = np.array([cam_dict[k] for k in keys])
    label_key = np.pad(np.array([k + 1 for k in keys]).astype(np.uint8), (1, 0), constant_values=0)  # 0 = bg
    cam = np.pad(cam, ((1, 0), (0, 0), (0, 0)), mode="constant", constant_values=t)              # bg channel = t
    cam = F.softmax(torch.tensor(cam).float(), dim=0).numpy()
    return label_key[np.argmax(cam, axis=0)]

def show_mask(ax, img, mask, title):
    ax.imshow(img); ax.imshow(CMAP[mask], alpha=0.6); ax.set_title(title, fontsize=9); ax.axis("off")

fig, axs = plt.subplots(2, 4, figsize=(18, 7))
for r, (img, d) in enumerate([(img_cats, cam_cats), (img_dog, cam_dog)]):
    for ax, t in zip(axs[r], [0.15, 0.30, 0.43, 0.65]):
        mask = cam_to_mask(d, t)
        names = [VOC_CLASSES[i - 1] for i in np.unique(mask) if i > 0]
        show_mask(ax, img, mask, f"t = {t}  (classes: {names})")
plt.suptitle("Pseudo masks from CAMs: low t -> objects too big, high t -> objects too small", fontsize=12)
plt.tight_layout(); plt.show()

## 4.2 CRF post-processing
A **dense CRF** (conditional random field, `tool/imutils.py` `crf_inference_label`) cleans the mask. It encourages neighbouring pixels with **similar colours** to share a label, which snaps the mask to object edges. The paper reports that CRF lifts the train-set CAM mIoU from **67.2%** to **69.4%** (Table IV), and the CRF-processed masks are what Phase 2 trains on.

`pydensecrf` must be compiled from source and often fails on Windows. The next cell uses it if it's installed and otherwise just explains what it would do.

In [ ]:
try:
    import pydensecrf.densecrf as dcrf
    from pydensecrf.utils import unary_from_labels

    def crf_inference_label(img, labels, t=10, n_labels=21, gt_prob=0.7):   # tool/imutils.py
        h, w = img.shape[:2]
        d = dcrf.DenseCRF2D(w, h, n_labels)
        d.setUnaryEnergy(unary_from_labels(labels, n_labels, gt_prob=gt_prob, zero_unsure=False))
        d.addPairwiseGaussian(sxy=4, compat=3)
        d.addPairwiseBilateral(sxy=83, srgb=5, rgbim=np.copy(img), compat=3)
        return np.argmax(np.array(d.inference(t)).reshape((n_labels, h, w)), axis=0)
    HAVE_CRF = True
except ImportError:
    HAVE_CRF = False
    print("pydensecrf not installed -> skipping the CRF demo.\n"
          "(Optional: `pip install git+https://github.com/lucasb-eyer/pydensecrf.git`, needs a C++ compiler.)")

if HAVE_CRF:
    fig, axs = plt.subplots(1, 4, figsize=(18, 3.6))
    for i, (img, d) in enumerate([(img_cats, cam_cats), (img_dog, cam_dog)]):
        keys = list(d.keys()); lab = cam_to_mask(d, 0.43)
        compact = np.zeros_like(lab); lut = {0: 0, **{k + 1: j + 1 for j, k in enumerate(keys)}}
        for a, b in lut.items(): compact[lab == a] = b                       # labels 0..len(keys)
        crf = crf_inference_label(np.array(img), compact, n_labels=len(keys) + 1)
        back = np.array([0] + [k + 1 for k in keys])[crf]
        show_mask(axs[2 * i], img, lab, "t=0.43, no CRF"); show_mask(axs[2 * i + 1], img, back, "t=0.43 + CRF")
    plt.tight_layout(); plt.show()

## 4.3 Measuring quality: mIoU

**IoU (Intersection over Union)** for one class is the overlap between the predicted region and the true region, divided by their union:

$$\mathrm{IoU}_c = \frac{|\text{pred}_c \cap \text{gt}_c|}{|\text{pred}_c \cup \text{gt}_c|} = \frac{TP}{TP + FP + FN}$$

**mIoU** is the mean IoU over the 21 classes (background included). Important: the repo **sums TP/FP/FN over the whole dataset first** and divides only at the end (`evaluation.py` L149-194). It does *not* average per-image scores. Pixels labelled 255 in the ground truth ("unsure", the object borders in VOC) are ignored.

A tiny example you can check by hand:

In [ ]:
def confusion_counts(pred, gt, num_cls=21):
    # evaluation.py compare(): counts per class P (predicted), T (true), TP; ignores gt == 255
    cal = gt < 255
    P = np.array([np.sum((pred == i) & cal) for i in range(num_cls)])
    Tt = np.array([np.sum((gt == i) & cal) for i in range(num_cls)])
    TP = np.array([np.sum((gt == i) & (pred == gt) & cal) for i in range(num_cls)])
    return P, Tt, TP

def miou_from_counts(P, Tt, TP):
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = TP / (Tt + P - TP)
    return np.nanmean(iou) * 100, iou

gt = np.array([[0, 0, 1, 1],
               [0, 0, 1, 1],
               [0, 0, 255, 1],
               [0, 0, 0, 0]])
pred = np.array([[0, 1, 1, 1],
                 [0, 0, 1, 1],
                 [0, 0, 1, 0],
                 [0, 0, 0, 0]])
m, iou = miou_from_counts(*confusion_counts(pred, gt, num_cls=2))
# The pixel marked 255 is ignored. Background: true 10, predicted 10, overlap 9 -> 9 / (10 + 10 - 9) = 0.818
# Class 1:    true 5,  predicted 5,  overlap 4 -> 4 / (5 + 5 - 4)   = 0.667
print("IoU background =", round(iou[0], 3))
print("IoU class 1    =", round(iou[1], 3))
print("mIoU           =", round(m, 1), "%")

## 4.4 (Optional) Reproduce the Phase-1 CAM numbers on PASCAL VOC

This needs the **PASCAL VOC 2012** images plus the **SegmentationClassAug** ground-truth masks (see Part 6). If you put them at `data/voc12/VOCdevkit/VOC2012/` next to this notebook (or inside `WeakTr/data/...`), the next cell will:

1. generate CAMs with the official checkpoint for the first `N_IMAGES` images of `voc12/train_id.txt` (the 1,464-image VOC *train* split used in Table I / IV);
2. sweep the background threshold $t$ like `evaluation.py --curve True`;
3. report mIoU. On all 1,464 images without CRF this should land near **67.2%** (Table IV, "WeakTr w/ AAF").

On a CPU expect ~5–10 s per image. `N_IMAGES = 100` gives a quick, noisier estimate.

In [ ]:
N_IMAGES = 100
cands = [ROOT / "data" / "voc12" / "VOCdevkit" / "VOC2012", REPO / "data" / "voc12" / "VOCdevkit" / "VOC2012"]
VOC = next((p for p in cands if (p / "SegmentationClassAug").exists()), None)
if VOC is None:
    print("VOC not found -> skipped. Expected one of:\n  " + "\n  ".join(map(str, cands)))
else:
    names = [l.strip() for l in open(REPO / "voc12" / "train_id.txt")][:N_IMAGES]
    thresholds = np.arange(0.30, 0.56, 0.01)
    counts = {t: [np.zeros(21), np.zeros(21), np.zeros(21)] for t in thresholds}
    t0 = time.time()
    for i, name in enumerate(names):
        img = load_image(VOC / "JPEGImages" / f"{name}.jpg")
        lbl = [VOC_CLASSES[j] for j in np.nonzero(cls_labels[name])[0]]
        cams = generate_cam(weaktr, img, lbl)
        gt = np.array(Image.open(VOC / "SegmentationClassAug" / f"{name}.png"))
        for t in thresholds:
            for acc, v in zip(counts[t], confusion_counts(cam_to_mask(cams, t), gt)): acc += v
        if (i + 1) % 20 == 0: print(f"{i + 1}/{len(names)} images, {time.time() - t0:.0f}s")
    curve = [miou_from_counts(*counts[t])[0] for t in thresholds]
    best = int(np.argmax(curve))
    plt.figure(figsize=(6, 3)); plt.plot(thresholds, curve, "o-"); plt.xlabel("background threshold t"); plt.ylabel("mIoU (%)")
    plt.title(f"CAM mIoU on {len(names)} VOC train images"); plt.show()
    print(f"best t = {thresholds[best]:.2f} -> mIoU = {curve[best]:.1f}%   (paper, all 1464 images, no CRF: 67.2%)")

---
# Part 5: Phase 2, online retraining with the gradient-clipping decoder

The pseudo masks from Part 4 are about **79% precise** (Table IV). Training a segmentation network on them directly also teaches it the mistakes. Earlier methods first ran an expensive refinement network (AffinityNet). WeakTr instead trains a segmentation network **directly** on the CAM masks and lets the network **skip the pixels it finds suspicious**.

The segmentation network is **Segmenter**: a ViT encoder plus a small transformer decoder (`OnlineRetraining/segm/model/`).

## 5.1 The decoder (`decoder.py` `MaskTransformer`, paper Eq. 10)
1. The ViT encoder produces patch tokens $T$ (for a 480×480 crop: 30×30 = 900 tokens).
2. The decoder appends **one learnable token per class** (21 including background) and runs **2 transformer layers** over everything.
3. Patch features and class features are both **L2-normalised**, and each patch's score for class $c$ is their **dot product** (cosine similarity), followed by a LayerNorm.
4. The 30×30×21 score map is **upsampled** to the image size.

$$\hat P = \mathrm{Upsample}\Big(\mathrm{LN}\big(\hat T_{norm}\,\hat Q_{norm}^\top\big)\Big)$$

In [ ]:
class SegBlock(nn.Module):        # segm/model/blocks.py Block (pre-norm transformer layer)
    def __init__(self, dim, heads, mlp_dim):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))[0]
        return x + self.mlp(self.norm2(x))

class MaskTransformer(nn.Module):  # segm/model/decoder.py L37-102
    def __init__(self, n_cls=21, patch_size=16, d_encoder=384, n_layers=2, n_heads=6, d_model=384, d_ff=1536):
        super().__init__()
        self.patch_size, self.n_cls, self.scale = patch_size, n_cls, d_model ** -0.5
        self.blocks = nn.ModuleList([SegBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.cls_emb = nn.Parameter(torch.randn(1, n_cls, d_model) * .02)         # one token per class
        self.proj_dec = nn.Linear(d_encoder, d_model)
        self.proj_patch = nn.Parameter(self.scale * torch.randn(d_model, d_model))
        self.proj_classes = nn.Parameter(self.scale * torch.randn(d_model, d_model))
        self.decoder_norm, self.mask_norm = nn.LayerNorm(d_model), nn.LayerNorm(n_cls)

    def forward(self, x, im_size):
        H, W = im_size; GS = H // self.patch_size
        x = torch.cat((self.proj_dec(x), self.cls_emb.expand(x.size(0), -1, -1)), 1)
        for blk in self.blocks: x = blk(x)
        x = self.decoder_norm(x)
        patches, cls_seg_feat = x[:, :-self.n_cls] @ self.proj_patch, x[:, -self.n_cls:] @ self.proj_classes
        patches = patches / patches.norm(dim=-1, keepdim=True)                      # L2 normalise
        cls_seg_feat = cls_seg_feat / cls_seg_feat.norm(dim=-1, keepdim=True)
        masks = self.mask_norm(patches @ cls_seg_feat.transpose(1, 2))              # Eq. (10) without upsampling
        return masks.reshape(x.size(0), GS, -1, self.n_cls).permute(0, 3, 1, 2)

dec = MaskTransformer()
fake_tokens = torch.randn(2, 30 * 30, 384)               # what a ViT-S encoder gives for a 480x480 crop
low_res = dec(fake_tokens, (480, 480))
full = F.interpolate(low_res, size=(480, 480), mode="bilinear")
print("encoder tokens", tuple(fake_tokens.shape), "-> decoder masks", tuple(low_res.shape), "-> upsampled", tuple(full.shape))
print(f"decoder parameters: {sum(p.numel() for p in dec.parameters()) / 1e6:.2f} M (encoder ViT-S: ~21.7 M)")

## 5.2 Why "loss" can stand in for "gradient"

The paper talks about each pixel's **gradient**, but the code clips the **per-pixel cross-entropy loss**. Are those the same thing?

For softmax + cross-entropy, the gradient with respect to the logits is $p - \text{onehot}(y)$. Its size grows whenever the probability $p_y$ of the labelled class falls, which is exactly when the loss $-\log p_y$ grows. So **"large loss" ⇔ "large gradient"**, and capping a pixel's loss at a threshold switches off that pixel's gradient. Let's check numerically:

In [ ]:
logits = torch.randn(5000, 21) * 3
labels = torch.randint(0, 21, (5000,))
logits.requires_grad_(True)
loss = F.cross_entropy(logits, labels, reduction="none")
loss.sum().backward()
grad_norm = logits.grad.norm(dim=1)
plt.figure(figsize=(5, 3.2))
plt.scatter(loss.detach(), grad_norm, s=2, alpha=.4)
plt.xlabel("per-pixel CE loss  -log p_y"); plt.ylabel("‖∂loss/∂logits‖"); plt.title("bigger loss ⇒ bigger gradient")
plt.show()
print("Spearman rank correlation:",
      round(np.corrcoef(loss.detach().argsort().argsort(), grad_norm.argsort().argsort())[0, 1], 3))

## 5.3 The gradient clipping rule (paper Eq. 11–15, `decoder.py` `GradientClipping`)

For a batch of predictions and CAM pseudo labels:

1. **Per-pixel loss** $\hat G$ = cross-entropy at every pixel (Eq. 11).
2. **Global constraint** $\lambda_{global}$ = mean loss over all pixels (Eq. 13).
3. **Start value $\tau$** (Eq. 15): if $\lambda_{global} > \tau$, the model is still too young to judge which labels are wrong, so **don't clip**. VOC uses $\tau = 1.2$ (`--start-value 1.2`).
4. **Local constraint** $\lambda_i$: cut the loss map into $S\times S$ patches (VOC: $S=120$ px on 480-px crops, `--patch-size 120`) and average each patch (Eq. 12).
5. **Threshold** per patch = $\max(\lambda_i, \lambda_{global})$. Pixels whose loss is **above** the threshold are masked out (Eq. 14).

In the code, step 5 is a neat trick: `loss = clamp(loss - thr, max=0) + thr` $=\min(\text{loss}, \text{thr})$. Since `thr` is detached, a clamped pixel still adds a constant to the loss but gives **zero gradient**.

Here is the repo's class, unchanged, plus a helper that also returns the keep/clip mask for visualisation:

In [ ]:
class GradientClipping(nn.Module):  # OnlineRetraining/segm/model/decoder.py L119-151 (verbatim)
    def __init__(self, start_value, patch_size):
        super().__init__()
        self.start_value = start_value
        self.patch_size = patch_size

    def forward(self, seg_pred, seg_gt, criterion):
        ori_loss = criterion(seg_pred, seg_gt)
        detach_loss = ori_loss.detach().clone()
        mean_loss = detach_loss.mean()                       # lambda_global

        # set start loss clamp threshold
        if mean_loss > self.start_value:                     # Eq. (15): too early -> no clipping
            return ori_loss, ori_loss

        b, h, w = detach_loss.shape
        # all batch average
        detach_loss = detach_loss.mean(dim=0).unsqueeze(0)
        local_mean = F.avg_pool2d(detach_loss.unsqueeze(1), kernel_size=self.patch_size,
                                  stride=self.patch_size, padding=h % self.patch_size,
                                  count_include_pad=False).squeeze(1)      # lambda_i, Eq. (12)
        local_mean = torch.maximum(local_mean, mean_loss)                  # max(lambda_i, lambda_global)
        local_mean = torch.repeat_interleave(local_mean, b, dim=0)
        local_mean = torch.repeat_interleave(local_mean, self.patch_size, dim=1)
        local_mean = torch.repeat_interleave(local_mean, self.patch_size, dim=2)

        clamp_loss = ori_loss - local_mean
        clamp_loss = torch.clamp(clamp_loss, None, 0)
        loss = clamp_loss + local_mean                                     # = min(loss, threshold), Eq. (14)

        return ori_loss, loss

def clip_with_mask(gc, pred, gt, criterion):
    ori, clipped = gc(pred, gt, criterion)
    kept = (ori.detach() <= clipped.detach() + 1e-12)       # True where the gradient survives
    return ori, clipped, kept

# A tiny worked example: one 4x4 "image", patch size 2
crit = nn.CrossEntropyLoss(reduction="none")
gt_toy = torch.zeros(1, 4, 4, dtype=torch.long); gt_toy[0, :2, 2:] = 1
pred_toy = torch.zeros(1, 2, 4, 4); pred_toy[0, 0] = 2.0; pred_toy[0, 1, :2, 2:] = 3.0
pred_toy[0, 1, 3, 0] = 4.0          # one pixel where the model strongly disagrees with the label
ori, clipped, kept = clip_with_mask(GradientClipping(start_value=1.2, patch_size=2), pred_toy, gt_toy, crit)
print("per-pixel loss:\n", ori[0].numpy().round(2))
print("\nglobal mean = %.3f" % ori.mean())
print("\nloss after clipping:\n", clipped[0].numpy().round(2))
print("\nkept (gradient flows):\n", kept[0].int().numpy())

## 5.4 A miniature experiment you can train on a CPU

Full Phase 2 trains a ViT-S on 10,582 images for 100 epochs (~9 h on a GPU). To see the *idea* at work, we build a tiny world instead:

* **Images**: 64×64 noise with a red **circle** (class 1) and/or a blue **square** (class 2).
* **True masks**: known exactly, so we can measure honestly.
* **Pseudo masks with CAM-like errors**: each object "bleeds" a random blob into the background (like CAMs spreading onto context), and part of each object is randomly **missing** (like CAMs covering only the most discriminative part).

We then train the same small CNN three ways, mirroring paper Table XIV:

| setting | meaning |
|---|---|
| no clipping ($\tau = 0$) | standard training on noisy labels |
| gradient clipping (moderate $\tau$) | WeakTr's rule |
| always clipping ($\tau = 999.9$) | clipping from the very first step |

Because the toy losses are smaller than VOC's, $\tau$ is rescaled (VOC: 1.2), and $S=16$ on 64-px images keeps the paper's ratio (120 px on 480 px).

In [ ]:
S_IMG = 64
def make_sample(rng):
    img = rng.normal(0.5, 0.12, (3, S_IMG, S_IMG)).astype(np.float32)
    gt = np.zeros((S_IMG, S_IMG), np.int64)
    yy, xx = np.mgrid[:S_IMG, :S_IMG]
    for cls in rng.permutation([1, 2])[: rng.integers(1, 3)]:
        r = rng.integers(8, 16); cy, cx = rng.integers(r, S_IMG - r, 2)
        m = ((yy - cy) ** 2 + (xx - cx) ** 2 < r * r) if cls == 1 else ((abs(yy - cy) < r) & (abs(xx - cx) < r))
        col = np.array([0.9, 0.3, 0.2] if cls == 1 else [0.2, 0.4, 0.9]) + rng.normal(0, .05, 3)
        img[:, m] = col[:, None] + rng.normal(0, 0.1, (3, m.sum())); gt[m] = cls
    pl = gt.copy()                                       # CAM-like pseudo label
    for cls in (1, 2):
        m = gt == cls
        if not m.any(): continue
        ys, xs = np.nonzero(m); i = rng.integers(len(ys)); r = rng.integers(5, 11)
        cy, cx = ys[i] + rng.integers(-8, 9), xs[i] + rng.integers(-8, 9)
        pl[((yy - cy) ** 2 + (xx - cx) ** 2 < r * r) & (gt == 0)] = cls      # bleeding into background
        i = rng.integers(len(ys)); r = rng.integers(4, 9)
        pl[((yy - ys[i]) ** 2 + (xx - xs[i]) ** 2 < r * r) & m] = 0           # missing object part
    return img, gt, pl

def make_dataset(n, seed):
    rng = np.random.default_rng(seed); s = [make_sample(rng) for _ in range(n)]
    return tuple(torch.tensor(np.stack([z[k] for z in s])) for k in range(3))

Xtr, Ytr, Ptr = make_dataset(400, 0)
Xte, Yte, _ = make_dataset(150, 1)
def miou3(pred, gt):
    return miou_from_counts(*confusion_counts(pred.numpy(), gt.numpy(), num_cls=3))[0]
print(f"pseudo labels vs truth (train): mIoU {miou3(Ptr, Ytr):.1f}%, pixel precision {(Ptr == Ytr).float().mean() * 100:.1f}%")

fig, axs = plt.subplots(3, 6, figsize=(13, 6.5))
for j in range(6):
    axs[0, j].imshow(Xtr[j].permute(1, 2, 0).clamp(0, 1)); axs[1, j].imshow(CMAP[Ytr[j].numpy()]); axs[2, j].imshow(CMAP[Ptr[j].numpy()])
for a, t in zip(axs[:, 0], ["image", "true mask", "pseudo (CAM-like)"]): a.set_ylabel(t)
for a in axs.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
class TinySegNet(nn.Module):   # a small U-Net; stands in for Segmenter so the experiment fits on a CPU
    def __init__(self, c=16):
        super().__init__()
        blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())
        self.e1, self.e2, self.e3 = blk(3, c), blk(c, 2 * c), blk(2 * c, 4 * c)
        self.d2, self.d1, self.out = blk(6 * c, 2 * c), blk(3 * c, c), nn.Conv2d(c, 3, 1)
    def forward(self, x):
        a = self.e1(x); b = self.e2(F.max_pool2d(a, 2)); c = self.e3(F.max_pool2d(b, 2))
        b = self.d2(torch.cat([F.interpolate(c, scale_factor=2), b], 1))
        return self.out(self.d1(torch.cat([F.interpolate(b, scale_factor=2), a], 1)))

def train_seg(start_value, epochs, seed=0, log_every=0):
    # mirrors segm/engine.py train_one_epoch: CE(reduction='none') -> GradientClipping -> mean -> step
    torch.manual_seed(seed)
    net = TinySegNet(); opt = torch.optim.Adam(net.parameters(), 2e-3)
    crit = nn.CrossEntropyLoss(reduction="none")
    gc = GradientClipping(start_value=start_value, patch_size=16)
    hist = {"miou": [], "kept_precision": [], "kept_frac": []}
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xtr)); kp, kf = [], []
        for i in range(0, len(Xtr), 16):
            idx = perm[i:i + 16]
            ori, loss, kept = clip_with_mask(gc, net(Xtr[idx]), Ptr[idx], crit)
            kp.append(((Ptr[idx] == Ytr[idx]) & kept).sum().item() / kept.sum().item())
            kf.append(kept.float().mean().item())
            opt.zero_grad(); loss.mean().backward(); opt.step()
        net.eval()
        with torch.no_grad(): hist["miou"].append(miou3(net(Xte).argmax(1), Yte))
        hist["kept_precision"].append(100 * np.mean(kp)); hist["kept_frac"].append(100 * np.mean(kf))
        if log_every and (ep + 1) % log_every == 0:
            print(f"  epoch {ep + 1:2d}: test mIoU {hist['miou'][-1]:.1f}%, precision of kept labels {hist['kept_precision'][-1]:.1f}%")
    return net, hist

### Look at one batch through the clipping rule
We first train briefly *without* clipping (the "young model"). Then we pass a batch through `GradientClipping` and look at which pixels it throws away, next to the pixels whose pseudo label is actually wrong.

In [ ]:
net_warm, _ = train_seg(start_value=0.0, epochs=4)        # tau=0 -> mean loss is always > tau -> never clips
net_warm.eval()
idx = torch.arange(8)
with torch.no_grad():
    ori, clipped, kept = clip_with_mask(GradientClipping(999.9, 16), net_warm(Xtr[idx]), Ptr[idx], nn.CrossEntropyLoss(reduction="none"))
wrong = (Ptr[idx] != Ytr[idx])
fig, axs = plt.subplots(5, 6, figsize=(13, 11))
for j in range(6):
    axs[0, j].imshow(Xtr[j].permute(1, 2, 0).clamp(0, 1))
    axs[1, j].imshow(CMAP[Ptr[j].numpy()])
    axs[2, j].imshow(ori[j], cmap="magma")
    axs[3, j].imshow(~kept[j], cmap="gray")
    axs[4, j].imshow(wrong[j], cmap="gray")
for a, t in zip(axs[:, 0], ["image", "pseudo label", "per-pixel loss Ĝ", "CLIPPED (white)", "pseudo label WRONG (white)"]):
    a.set_ylabel(t)
for a in axs.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
p_kept = (~wrong & kept).sum().item() / kept.sum().item(); p_all = (~wrong).float().mean().item()
print(f"label precision on ALL pixels: {100 * p_all:.1f}%  ->  on KEPT pixels: {100 * p_kept:.1f}%  "
      f"(clipped {100 * (~kept).float().mean():.1f}% of pixels)")
print(f"of the wrong pixels, {100 * (~kept & wrong).sum() / wrong.sum():.0f}% were clipped; "
      f"of the correct pixels, {100 * (~kept & ~wrong).sum() / (~wrong).sum():.0f}% were clipped")

The white regions in rows 4 and 5 overlap a lot. **High-loss pixels are mostly the wrong labels.** This is the paper's Fig. 3 observation ("pixels with smaller gradients are associated with more accurate pseudo labels"). It also shows a cost: some correct pixels near object **boundaries** get clipped too (paper Sec. V-E discusses this).

### The full comparison (≈ 5–10 minutes on CPU)

In [ ]:
EPOCHS = 25
TAU = 0.3          # toy-scale start value (VOC uses 1.2 with 21 classes)
results = {}
for name, tau in [("no clipping (τ=0)", 0.0), (f"gradient clipping (τ={TAU})", TAU), ("always clipping (τ=999.9)", 999.9)]:
    t0 = time.time(); print(name)
    _, results[name] = train_seg(tau, EPOCHS, log_every=5)
    print(f"  ({time.time() - t0:.0f}s)")
fig, axs = plt.subplots(1, 3, figsize=(17, 3.8))
for name, h in results.items():
    axs[0].plot(h["miou"], label=name); axs[1].plot(h["kept_precision"], label=name); axs[2].plot(h["kept_frac"], label=name)
axs[0].axhline(miou3(Ptr, Ytr), ls="--", c="k", lw=.8, label="mIoU of the pseudo labels themselves")
axs[0].set_title("test mIoU vs TRUE masks"); axs[1].set_title("precision of labels that receive gradient (cf. Table XIII)")
axs[2].set_title("% of pixels that receive gradient")
for a in axs: a.set_xlabel("epoch"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()
for name, h in results.items():
    print(f"{name:32s} final test mIoU {h['miou'][-1]:.1f}% | best {max(h['miou']):.1f}% | kept-label precision {h['kept_precision'][-1]:.1f}%")

RESULT_DISCUSSION

## 5.5 Where this lands in the paper

| Paper result (VOC12 val, DeiT-S) | mIoU |
|---|---|
| naive decoder (no clipping) | 71.5 (Table VII); 71.8 with $\tau=0$ (Table XIV) |
| gradient clipping, $\tau = 1.2$ | 73.0 |
| gradient clipping, $\tau = 1.2$ + CRF at inference | **74.0** |
| always clipping, $\tau = 999.9$ | 54.2 |
| patch size $S$: 480 / 240 / 160 / **120** / 96 | 73.3 / 73.3 / 73.5 / **74.0** / 73.3 (Table VIII) |
| with a stronger encoder (ImageNet-21k ViT-S) | **78.4** (Table IX) |

During real training, the precision of the pixels that keep their gradient rises from 78.9% (the raw CAM masks) to **91.2%** after 40 epochs (Table XIII). That is the same effect as the middle plot above.

---
# Part 6: Reproducing the paper's numbers at full scale

Everything above runs on a laptop. Reproducing the tables requires the real training runs. Here is exactly what that takes.

### 6.1 Hardware and environment
* **NVIDIA GPU with CUDA** (the authors used one TITAN X: Phase 1 ≈ 2h40, Phase 2 ≈ 9h, Table XI). ≥ 12 GB VRAM is comfortable.
* **Linux** (or **WSL2** on Windows, or Google Colab / Kaggle). The scripts call `bash dist_train.sh` and `torch.distributed`, and `pydensecrf` / `mmcv-full` / `apex` are painful to build on native Windows.
* Environment (`docs/Install.md`):
```bash
conda create -n weaktr python=3.7 -y && conda activate weaktr
pip install torch==1.8.0+cu111 torchvision==0.9.0+cu111 --extra-index-url https://download.pytorch.org/whl/cu111
pip install -r requirements.txt            # timm==0.5.4, scikit-learn, opencv, mlflow, pycocotools, einops, tqdm
pip install -U openmim && mim install mmcv-full==1.4.0 && pip install mmsegmentation==0.30.0
pip install git+https://github.com/lucasb-eyer/pydensecrf.git
# Phase-2 blocks.py imports xformers (docs: "Environment Update", torch 1.12.1+cu116, xformers==0.0.20, apex)
```

### 6.2 Data (`docs/prepare_dataset.md`, `data/download_and_convert_voc12.sh`)
```
WeakTr/data/voc12/VOCdevkit/VOC2012/
    JPEGImages/               17,125 images (VOCtrainval_11-May-2012.tar, ~2 GB)
    SegmentationClass/        official masks
    SegmentationClassAug/     10,582 "train_aug" masks from SBD (needed for evaluation and Phase 2)
```
Image-level labels are already in the repo (`voc12/cls_labels.npy`), as are the split lists (`train_id.txt` = 1,464, `train_aug_id.txt` = 10,582, `val_id.txt` = 1,449).

### 6.3 The commands, with what each one does (run from `WeakTr/`)

**Phase 1a: train the CAM network** (≈ 60 epochs; `checkpoint_best_mIoU.pth` is the epoch whose CAMs score best on `train_id.txt`)
```bash
python main.py --model deit_small_WeakTr_patch16_224 --data-path data --data-set VOC12 \
    --img-ms-list voc12/train_id.txt --cam-npy-dir WeakTr_results/WeakTr/attn-patchrefine-npy \
    --output_dir WeakTr_results/WeakTr --reduction 8 --pool-type max --lr 6e-4 --weight-decay 0.03
```
**Phase 1b: generate multi-scale CAMs** for all 10,582 train_aug images (Section 3.10 of this notebook)
```bash
python main.py --model deit_small_WeakTr_patch16_224 --data-path data --data-set VOC12MS \
    --img-ms-list voc12/train_aug_id.txt --scales 1.0 1.2 --gen_attention_maps \
    --cam-npy-dir WeakTr_results/WeakTr/attn-patchrefine-npy-ms --output_dir WeakTr_results/WeakTr \
    --resume WeakTr_results/WeakTr/checkpoint_best_mIoU.pth --reduction 8 --pool-type max
```
**Phase 1c: evaluate the CAMs (threshold sweep)** → **Table IV: 67.2%**
```bash
python evaluation.py --list voc12/train_id.txt --data-path data --type npy \
    --predict_dir WeakTr_results/WeakTr/attn-patchrefine-npy-ms --curve True --start 40
```
**Phase 1d: CRF → pseudo masks for Phase 2** → **Table IV: 69.4%**
```bash
python evaluation.py --list voc12/train_aug_id.txt --data-path data --type npy \
    --predict_dir WeakTr_results/WeakTr/attn-patchrefine-npy-ms \
    --out-dir WeakTr_results/WeakTr/pseudo-mask-ms-crf --t 43 --out-crf
```
**Phase 2: online retraining with gradient clipping** (`--ann-dir` points at the CRF pseudo masks)
```bash
cd OnlineRetraining
bash segm/dist_train.sh 1 --log-dir start1.2_patch120_seg_deit_small_patch16_224_voc_weaktr \
    --dataset pascal_voc --backbone deit_small_patch16_224 --decoder mask_transformer \
    --ann-dir WeakTr_CAMlb_wCRF --start-value 1.2 --patch-size 120
```
**Evaluation: multi-scale + CRF on val** → **Table II: 74.0%** (DeiT-S), **78.4%** with `--backbone vit_small_patch16_384`
```bash
bash segm/dist_test.sh 1 --multiscale --predict-dir <log-dir>/seg_prob_ms <log-dir>/checkpoint.pth pascal_voc
python -m segm.eval.make_crf --list val.txt --data-path ../data --predict-dir <log-dir>/seg_prob_ms \
    --predict-png-dir <log-dir>/seg_pred_ms --num-cls 21 --dataset voc12
```

### 6.4 Shortcuts if compute is limited
The README provides every intermediate artefact, so you can reproduce **one stage at a time**:

| Skip | Download (README table) | Then run | Expected |
|---|---|---|---|
| Phase 1 training | Step1 *Checkpoint* (already used in this notebook) | Phase 1b → 1c | 67.2% train CAM mIoU |
| All of Phase 1 | Step1 *CAM_Label* (CRF pseudo masks) | evaluate them against `SegmentationClassAug` with `evaluation.py --type png` | 69.4% |
| Phase 2 training | Step2 *Checkpoint* | the evaluation commands above | 74.0% val (DeiT-S) |

The cheapest *real* reproduction is the second row: no GPU at all, just the VOC ground truth and `evaluation.py`.

### 6.5 Things that commonly go wrong
* **Linear LR scaling**: `main.py` multiplies `--lr` by `batch_size × GPUs / 512`. If you change the batch size, the effective LR changes too.
* **Model selection uses train-set masks.** Phase 1 picks the checkpoint by CAM mIoU on `train_id.txt` with the ground-truth masks. This is common practice in WSSS papers for evaluating CAMs, but be aware of it.
* `segm/model/utils.py resize_pos_embed` calls `.cuda()`, so Phase 2 cannot run on CPU without editing it.
* Results vary by roughly ±0.5 mIoU between runs; the seed is fixed (`--seed 504`), but cuDNN and AMP are not fully deterministic.

---
# Part 7: Summary

| | Phase 1: CAM generation | Phase 2: online retraining |
|---|---|---|
| **Input** | images + image-level labels | images + CAM pseudo masks (after CRF) |
| **Network** | DeiT-S with 20 class tokens, conv head, **AAF** | Segmenter: ViT encoder + mask-transformer decoder |
| **Key idea** | not all 72 attention heads are equally useful, so **learn a weight per head** (Eq. 3–5), trained only by $L_{fine}$ | high-loss pixels are probably mislabelled, so **cap the loss at max(local mean, global mean)** once the mean loss falls below $\tau$ (Eq. 11–15) |
| **Output** | fine CAM → threshold $t$=0.43 → CRF → pseudo masks (69.4% mIoU on train) | final segmentation model (74.0% val mIoU with DeiT-S, 78.5% with EVA-02-S) |

### Exercises to deepen your understanding
1. In Section 3.8, change `"cat"` to `"sofa"`, or use the dog image with `"car"`. Do the same heads get high weights for every class and image?
2. In Section 3.9, replace the AAF weights with `torch.ones(72)` inside `fuse()` and build a "mean-sum" fine CAM. How do the maps differ?
3. In Section 4.1, which threshold $t$ gives the best-looking mask for each image? Why might one global $t$ be a compromise?
4. In Section 5.4, change the patch size `16` to `64` (one patch = whole image, i.e. global constraint only) or `8`. Compare with paper Table VIII.
5. Make the pseudo-label noise **systematic** (e.g. always bleed to the right of the object) instead of random. Does gradient clipping still help? Why or why not?